# 00 · Data Audit — Component 3 (Corneal Cellular Intelligence Platform)

**Phase 0, Step 0.5.** Read-only analysis of the NEBSL specular-microscopy dataset and its
Konan CellChek D ground truth. Nothing here changes data or splits.

| # | Section | Output (in `outputs/audit/`) |
|---|---------|------------------------------|
| 1 | Dataset counts & exclusions | `01_counts.csv` |
| 2 | Ground-truth distributions | `02_gt_distributions.png`, `02_gt_summary.csv` |
| 3 | Biomarker relationships | `03_relationships.png` |
| 4 | Grade / eye / year mix | `04_metadata_mix.png` |
| 5 | Split balance | `05_split_balance.png`, `05_split_balance.csv` |
| 6 | Image quality metrics | `06_image_quality.png`, `06_image_quality.csv` |
| 7 | Stress lines (exploratory ridge detector — not used downstream) | `07_*.png`, `07_stress_lines.csv` |
| 8 | Example gallery (best / typical / worst) | `08_gallery.png` |
| 9 | Cell size from FFT | `09_cell_spacing.png` |
| 10 | Audit summary | `audit_summary.json` |

**Run on Colab:** Runtime → Run all. A CPU runtime is enough (no GPU needed).

## 0 · Setup

In [ ]:
# Works on Colab (mounts Drive, pulls the repo, copies images to fast disk) and locally.
import os, sys, subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RESEARCH = "/content/drive/MyDrive/Colab Notebooks/Research_Project"
    REPO_DIR = f"{RESEARCH}/xai-cornea-grading"
    LOCAL_DATA = "/content/data"

    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=False)

    # Copy images once per session — reading from local disk is much faster than Drive.
    if not Path(LOCAL_DATA, "Finalized_images").exists():
        os.makedirs(LOCAL_DATA, exist_ok=True)
        subprocess.run(["cp", "-r", f"{RESEARCH}/Finalized_images", LOCAL_DATA], check=True)
        subprocess.run(["cp", f"{RESEARCH}/validation_ground_truth_dataset.xlsx", LOCAL_DATA], check=True)

    os.environ["CELLULAR_DATA_ROOT"] = LOCAL_DATA
    os.environ["CELLULAR_OUTPUT_ROOT"] = f"{RESEARCH}/cellular_outputs"
    os.environ["CELLULAR_MODELS_ROOT"] = f"{RESEARCH}/cellular_models"
    COMPONENT_DIR = Path(REPO_DIR, "ml", "cellular")
else:
    COMPONENT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(COMPONENT_DIR)
if str(COMPONENT_DIR) not in sys.path:
    sys.path.insert(0, str(COMPONENT_DIR))
print("Component dir:", COMPONENT_DIR)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from scipy import stats
from tqdm.auto import tqdm

from config import load_config

cfg = load_config()
P = cfg["paths"]
OUT = P["outputs_dir"] / "audit"
OUT.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
SPLIT_ORDER = ["train", "calibration", "val", "test"]
SEED = cfg["split"]["seed"]


def save(fig, name):
    """Save a figure to outputs/audit/ and show it."""
    fig.savefig(OUT / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()


def load_gray(image_file):
    """Read one specular image as a uint8 grayscale array."""
    return cv2.imread(str(P["images_dir"] / image_file), cv2.IMREAD_GRAYSCALE)


print("Images :", P["images_dir"])
print("Outputs:", OUT)

## 1 · Dataset counts & exclusions
Every image and ground-truth row is accounted for; exclusions carry a reason (from `splits/manifest.csv`).

In [ ]:
man = pd.read_csv(P["splits_dir"] / "manifest.csv")
split_summary = json.loads((P["splits_dir"] / "split_summary.json").read_text())
for col in ("donor_age", "year", "ecd", "cv", "hex", "mean_area", "num_cells"):
    man[col] = pd.to_numeric(man[col], errors="coerce")

inc = man[man["status"] == "included"].copy()

counts = pd.Series({
    "image files": int((man["status"] != "excluded:missing_image").sum()),
    "ground-truth rows": int(man["cornea_id"].notna().sum()),
    "included": len(inc),
    "needs_review (held out)": int((man["status"] == "review:needs_review").sum()),
    "excluded": int(man["status"].str.startswith("excluded").sum()),
    "unique tissues (included)": int(inc["group_id"].nunique()),
    "tissues with >1 capture": int((inc.groupby("group_id").size() > 1).sum()),
}, name="count")
counts.to_csv(OUT / "01_counts.csv")

display(counts.to_frame())
display(man["status"].value_counts().to_frame("images"))
display(man.loc[man["status"] != "included", ["image_file", "status"]])

## 2 · Ground-truth distributions
Konan CellChek D values for included images. Red dashed lines: clinical thresholds from the scope
(ECD < 2000 = poor viability; HEX < 60% = below healthy).

In [ ]:
BIOMARKERS = {"ecd": "ECD (cells/mm²)", "cv": "CV (%)", "hex": "HEX (%)",
              "mean_area": "Mean cell area (µm²)", "num_cells": "Cells analysed by Konan"}
REF_LINES = {"ecd": [2000], "hex": [60]}

fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, (col, label) in zip(axes, BIOMARKERS.items()):
    sns.histplot(inc[col], bins=40, ax=ax, color="tab:blue")
    for x in REF_LINES.get(col, []):
        ax.axvline(x, color="tab:red", ls="--", lw=1)
    ax.set_xlabel(label)
    ax.set_ylabel("images")
fig.suptitle("Konan ground-truth distributions (included images)")
save(fig, "02_gt_distributions")

gt_desc = inc[list(BIOMARKERS)].describe(percentiles=[.05, .25, .5, .75, .95]).T.round(1)
gt_desc.to_csv(OUT / "02_gt_summary.csv")
display(gt_desc)
print(f"ECD < 2000 (poor viability): {(inc['ecd'] < 2000).mean():.1%}")
print(f"HEX < 60%  (below healthy):  {(inc['hex'] < 60).mean():.1%}")
print(f"Konan cells analysed per image: median {inc['num_cells'].median():.0f}")

## 3 · Biomarker relationships
* **ECD vs mean area** should follow ECD = 10⁶ / mean area (Konan computes ECD from the analysed cell area).
  This matters for Phase 4–5: ECD accuracy depends directly on cell-area accuracy and the µm/px scale.
* **CV vs HEX** — polymegethism and pleomorphism usually move together.
* **ECD vs donor age** — ECD is expected to decline with age.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.scatter(inc["mean_area"], inc["ecd"], s=6, alpha=.5)
xs = np.linspace(inc["mean_area"].min(), inc["mean_area"].max(), 200)
ax.plot(xs, 1e6 / xs, "r-", lw=1.5, label="ECD = 10⁶ / mean area")
ax.set(xlabel="Mean cell area (µm²)", ylabel="ECD (cells/mm²)", title="ECD vs mean area")
ax.legend()

ax = axes[1]
r_cv_hex, _ = stats.pearsonr(inc["cv"], inc["hex"])
ax.scatter(inc["cv"], inc["hex"], s=6, alpha=.5)
ax.set(xlabel="CV (%)", ylabel="HEX (%)", title=f"CV vs HEX  (r = {r_cv_hex:.2f})")

ax = axes[2]
d = inc.dropna(subset=["donor_age"])
age_fit = stats.linregress(d["donor_age"], d["ecd"])
ages = np.linspace(d["donor_age"].min(), d["donor_age"].max(), 100)
ax.scatter(d["donor_age"], d["ecd"], s=6, alpha=.5)
ax.plot(ages, age_fit.intercept + age_fit.slope * ages, "r-")
ax.set(xlabel="Donor age (years)", ylabel="ECD (cells/mm²)",
       title=f"ECD vs age  (slope {age_fit.slope:.1f}/yr, r = {age_fit.rvalue:.2f})")
save(fig, "03_relationships")

ecd_area_diff = ((inc["ecd"] - 1e6 / inc["mean_area"]).abs() / inc["ecd"])
print(f"|ECD − 10⁶/mean_area| / ECD : median {ecd_area_diff.median():.2%}, "
      f"95th pct {ecd_area_diff.quantile(.95):.2%}, max {ecd_area_diff.max():.2%}")

## 4 · Grade, eye and year mix
Konan reliability grade (green / yellow / red), eye (OD = right, OS = left) and acquisition year.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["grade", "eye", "year"]):
    values = inc[col].astype("Int64") if col == "year" else inc[col]   # 2020, not 2020.0
    vc = values.astype("string").fillna("missing").value_counts().sort_index()
    ax.bar(vc.index, vc.values)
    ax.set(title=col, ylabel="images")
    for i, v in enumerate(vc.values):
        ax.text(i, v, str(v), ha="center", va="bottom")
save(fig, "04_metadata_mix")

## 5 · Split balance
The frozen tissue-level split should have similar biomarker distributions in every split.
KS test p-values > 0.05 mean no detectable difference between train and test.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, col in zip(axes, ["ecd", "cv", "hex"]):
    sns.kdeplot(data=inc, x=col, hue="split", hue_order=SPLIT_ORDER, common_norm=False, ax=ax)
    ax.set_xlabel(BIOMARKERS[col])
fig.suptitle("Biomarker distributions per split")
save(fig, "05_split_balance")

balance = (inc.groupby("split")[["ecd", "cv", "hex", "mean_area"]]
              .agg(["mean", "std"]).round(1).reindex(SPLIT_ORDER))
balance.insert(0, ("n", "images"), inc["split"].value_counts().reindex(SPLIT_ORDER))
balance.to_csv(OUT / "05_split_balance.csv")
display(balance)

ks_pvalues = {}
for col in ["ecd", "cv", "hex"]:
    ks = stats.ks_2samp(inc.loc[inc["split"] == "train", col], inc.loc[inc["split"] == "test", col])
    ks_pvalues[col] = round(float(ks.pvalue), 4)
print("KS train vs test p-values:", ks_pvalues)

## 6 · Image quality metrics
Computed for every included image (about 1 minute):

| Metric | Meaning |
|--------|---------|
| `brightness`, `contrast` | mean and std of pixel intensity |
| `sharpness` | variance of the Laplacian — low = blurred |
| `illum_unevenness` | std / mean of the smooth illumination field — high = uneven lighting |
| `dark_fraction` | share of the image whose illumination is very dark (< 40) |
| `textured_fraction` | share of the image with clear cell-scale texture — a proxy for analysable area |
| `illumination_falloff` | depth of the strongest dark zone in the row / column brightness profile |

*Note:* `illumination_falloff` was first tried as a stress-line detector but it measures the normal
bright-centre / dark-edge lighting of specular images (it flagged 86% of images). Stress lines are
detected properly in Section 7.

`quality_score` (0–100) ranks images by sharpness + textured area − unevenness − darkness.
It is a descriptive audit measure only; the real ROI quality model is built in Phase 6.

In [ ]:
def illumination_falloff(background):
    """
    Depth of the darkest zone in the row / column brightness profile, relative
    to its median, ignoring a 10% margin. High = strong uneven illumination.
    """
    norm = background / (np.median(background) + 1e-6)
    scores = []
    for axis in (0, 1):
        prof = norm.mean(axis=axis).astype(np.float32)
        prof = cv2.GaussianBlur(prof.reshape(1, -1), (0, 0), 2).ravel()
        n = len(prof)
        core = prof[int(.1 * n): int(.9 * n)]
        scores.append(1.0 - core.min() / (np.median(prof) + 1e-6))
    return float(max(scores))


def quality_metrics(img):
    f = img.astype(np.float32)
    small = cv2.resize(f, None, fx=0.25, fy=0.25, interpolation=cv2.INTER_AREA)
    background = cv2.GaussianBlur(small, (0, 0), 12)            # illumination field
    texture = f - cv2.GaussianBlur(f, (0, 0), 6)                 # cell-scale detail
    tex_energy = np.sqrt(cv2.GaussianBlur(texture ** 2, (0, 0), 16))
    return {
        "brightness": float(f.mean()),
        "contrast": float(f.std()),
        "sharpness": float(cv2.Laplacian(f, cv2.CV_32F).var()),
        "illum_unevenness": float(background.std() / (background.mean() + 1e-6)),
        "dark_fraction": float((background < 40).mean()),
        "textured_fraction": float((tex_energy > 0.5 * np.percentile(tex_energy, 95)).mean()),
        "illumination_falloff": illumination_falloff(background),
    }


rows = []
for _, r in tqdm(inc.iterrows(), total=len(inc), desc="Quality metrics"):
    rows.append({"image_id": r["image_id"], **quality_metrics(load_gray(r["image_file"]))})
quality = pd.DataFrame(rows)

def z(s):
    return (s - s.mean()) / (s.std() + 1e-9)

raw_score = (z(np.log1p(quality["sharpness"])) + z(quality["textured_fraction"])
             - z(quality["illum_unevenness"]) - z(quality["dark_fraction"]))
quality["quality_score"] = (raw_score.rank(pct=True) * 100).round(1)

inc = inc.merge(quality, on="image_id", how="left")
quality.to_csv(OUT / "06_image_quality.csv", index=False)

QUALITY_COLS = ["brightness", "contrast", "sharpness", "illum_unevenness",
                "dark_fraction", "textured_fraction"]
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, col in zip(axes.ravel(), QUALITY_COLS):
    sns.histplot(inc[col], bins=40, ax=ax)
fig.suptitle("Image quality metrics (included images)")
save(fig, "06_image_quality")

display(inc[QUALITY_COLS + ["illumination_falloff", "quality_score"]].describe().T.round(3))

# Does image quality relate to the Konan measurements?
for col in ["ecd", "num_cells", "hex"]:
    rho, p = stats.spearmanr(inc["quality_score"], inc[col])
    print(f"Spearman quality_score vs {col:9s}: rho = {rho:+.2f} (p = {p:.3f})")

## 7 · Stress lines — ridge detector (EXPLORATORY, not used downstream)

> **Conclusion from the 2026-10-08 run:** the detector flagged 39% of images, but on visual review
> many red lines traced the **edge of the illuminated area** or short debris, and several clear stress
> lines were missed (≈55% agreement). Stress lines vary too much (bright/dark, wide/thin, faint/strong)
> for a classical filter to separate them reliably from illumination boundaries.
> **Artifact presence is therefore learned from expert patch labels** in Sub-component 3.1, and its
> prevalence is measured from the Phase 1 annotation ("artifact" tag) — not from this heuristic.
> The section is kept as documented evidence for that design decision.

Stress lines / folds are **thin, long, curved lines** crossing the cell mosaic — much wider than a
cell wall and much longer than a cell. Detection tried:

1. Work at half resolution and **flatten the illumination** (divide by a smooth background), so the
   bright-centre / dark-edge lighting no longer matters; ignore very dark areas.
2. **Sato ridge filter** at scales wider than cell walls (bright and dark lines both count).
3. Thresholds are taken from a calibration sample of the dataset (hysteresis: strong seeds, weak extension).
4. Skeletonise and keep only **long** ridges (≥ `MIN_LINE_LENGTH_PX`); short ones are debris / noise.
5. `stress_line_length` = total length of long ridges (full-resolution px). An image is **flagged**
   when it exceeds `STRESS_LINE_MIN_TOTAL_PX`.

The galleries draw the detected lines in **red** so you can judge them. Count the mistakes and enter
them in the next cell; if agreement is poor, tune the constants and re-run this section.
Runtime ≈ 5–10 min on a CPU runtime.

In [ ]:
from skimage.filters import sato, apply_hysteresis_threshold
from skimage.morphology import skeletonize
from skimage.measure import label

RIDGE_SCALE = 0.5                 # work at half resolution (cells ≈ 14 px)
RIDGE_SIGMAS = (4, 6, 8)          # half-res px — wider than cell walls, matches stress-line width
DARK_BACKGROUND = 40              # ignore regions whose illumination is darker than this
MIN_LINE_LENGTH_PX = 150          # full-res px; shorter ridges are discarded
STRESS_LINE_MIN_TOTAL_PX = 300    # full-res px of long ridges needed to flag an image
CALIB_N = 80                      # images used to set the ridge thresholds
N_SHOW = 12


def ridge_response(img):
    """Sato ridge response of the illumination-flattened image (half resolution)."""
    f = cv2.resize(img.astype(np.float32), None, fx=RIDGE_SCALE, fy=RIDGE_SCALE,
                   interpolation=cv2.INTER_AREA)
    bg = cv2.GaussianBlur(f, (0, 0), 25)
    valid = bg > DARK_BACKGROUND
    flat = np.where(valid, f / (bg + 1.0), 1.0)
    resp = np.maximum(sato(flat, sigmas=RIDGE_SIGMAS, black_ridges=True),
                      sato(flat, sigmas=RIDGE_SIGMAS, black_ridges=False))
    # drop the border of the valid area: the step into darkness is not a stress line
    core = cv2.erode(valid.astype(np.uint8), np.ones((15, 15), np.uint8)).astype(bool)
    resp[~core] = 0
    return resp, f


# Dataset-level thresholds from a calibration sample (same rule for every image).
calib = inc.sample(min(CALIB_N, len(inc)), random_state=SEED)
pooled = np.concatenate([ridge_response(load_gray(f))[0].ravel()[::7]
                         for f in tqdm(calib["image_file"], desc="Ridge thresholds")])
pooled = pooled[pooled > 0]
RIDGE_LOW, RIDGE_HIGH = np.percentile(pooled, [97, 99.5])
print(f"Ridge thresholds: low {RIDGE_LOW:.4f}, high {RIDGE_HIGH:.4f}")


def detect_stress_lines(img):
    """Return (total long-ridge length in full-res px, number of long ridges, line mask, small image)."""
    resp, small = ridge_response(img)
    skel = skeletonize(apply_hysteresis_threshold(resp, RIDGE_LOW, RIDGE_HIGH))
    lab = label(skel, connectivity=2)
    lengths = np.bincount(lab.ravel())[1:] / RIDGE_SCALE          # skeleton px -> full-res px
    keep = np.flatnonzero(lengths >= MIN_LINE_LENGTH_PX) + 1
    line_mask = np.isin(lab, keep)
    return float(lengths[keep - 1].sum()), int(len(keep)), line_mask, small


rows = []
for _, r in tqdm(inc.iterrows(), total=len(inc), desc="Stress lines"):
    total, n_lines, _, _ = detect_stress_lines(load_gray(r["image_file"]))
    rows.append({"image_id": r["image_id"], "stress_line_length": total, "n_stress_lines": n_lines})
stress = pd.DataFrame(rows)
inc = inc.drop(columns=[c for c in stress.columns if c != "image_id" and c in inc], errors="ignore")
inc = inc.merge(stress, on="image_id", how="left")
inc["stress_line_flag"] = inc["stress_line_length"] >= STRESS_LINE_MIN_TOTAL_PX
stress.to_csv(OUT / "07_stress_lines.csv", index=False)

print(f"Flagged with stress lines: {inc['stress_line_flag'].sum()} of {len(inc)} "
      f"({inc['stress_line_flag'].mean():.1%})")
rho, p = stats.spearmanr(inc["stress_line_length"], inc["num_cells"])
print(f"Spearman stress_line_length vs Konan num_cells: rho = {rho:+.2f} (p = {p:.3f})")


def gallery(df, title, name, ncols=6):
    """Thumbnails with detected stress lines drawn in red."""
    nrows = max(1, int(np.ceil(len(df) / ncols)))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 2.6 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for ax, (_, r) in zip(axes, df.iterrows()):
        total, n_lines, mask, small = detect_stress_lines(load_gray(r["image_file"]))
        rgb = np.dstack([np.clip(small, 0, 255).astype(np.uint8)] * 3)
        thick = cv2.dilate(mask.astype(np.uint8), np.ones((3, 3), np.uint8)).astype(bool)
        rgb[thick] = (255, 0, 0)
        ax.imshow(rgb)
        ax.set_title(f"{r['image_id'][:22]}\n{n_lines} lines · {total:.0f} px", fontsize=8)
    fig.suptitle(title)
    save(fig, name)


flagged_sample = inc[inc["stress_line_flag"]].sample(
    min(N_SHOW, int(inc["stress_line_flag"].sum())), random_state=SEED)
clean_sample = inc[~inc["stress_line_flag"]].sample(
    min(N_SHOW, int((~inc["stress_line_flag"]).sum())), random_state=SEED)
gallery(flagged_sample, "FLAGGED — red lines should follow real stress lines / folds", "07_flagged_sample")
gallery(clean_sample, "NOT flagged — should have no (long) stress lines", "07_clean_sample")

In [ ]:
# Enter your visual check here (count from the two galleries above).
WRONGLY_FLAGGED = None   # flagged images that actually have NO stress line
MISSED = None            # not-flagged images that actually DO have a stress line

if WRONGLY_FLAGGED is None or MISSED is None:
    print("Fill in WRONGLY_FLAGGED and MISSED after looking at the galleries, then re-run this cell.")
    visual_agreement = None
else:
    n_checked = len(flagged_sample) + len(clean_sample)
    visual_agreement = 1 - (WRONGLY_FLAGGED + MISSED) / n_checked
    print(f"Heuristic agreement with visual check: {visual_agreement:.0%} "
          f"({n_checked} images checked)")

## 8 · Example gallery — best, typical and worst images (by `quality_score`)

In [ ]:
ranked = inc.sort_values("quality_score")
picks = pd.concat([
    ranked.tail(3).assign(group="best"),
    ranked.iloc[len(ranked) // 2 - 1: len(ranked) // 2 + 2].assign(group="typical"),
    ranked.head(3).assign(group="worst"),
])
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
for ax, (_, r) in zip(axes.ravel(), picks.iterrows()):
    ax.imshow(load_gray(r["image_file"]), cmap="gray")
    ax.set_title(f"{r['group'].upper()} · q={r['quality_score']:.0f} · ECD {r['ecd']:.0f} · "
                 f"HEX {r['hex']:.0f}\n{r['image_id']}", fontsize=9)
    ax.axis("off")
fig.suptitle("Image quality examples")
save(fig, "08_gallery")

## 9 · Cell size from the FFT
Endothelial cells form a roughly hexagonal lattice, which gives a ring in the 2-D power spectrum.
The ring radius gives the lattice row spacing; for a hexagonal lattice the cell centre-to-centre
distance is `d = 2 · spacing / √3`.

This confirms the cell size in **pixels**, which decides Phase 2 crop sizes. On the **calibration
split only** it also gives a *preliminary* µm/px estimate (hexagon area A = (√3/2)·d², using Konan's
mean area) — the real calibration is Phase 4.

In [ ]:
CROP = 512
FFT_SAMPLE = 200


def radial_power(img, crop=CROP):
    """Radially averaged, 1/f-compensated power spectrum of the most textured crop."""
    f = img.astype(np.float32)
    f = f - cv2.GaussianBlur(f, (0, 0), 10)                        # remove illumination
    energy = cv2.GaussianBlur(f ** 2, (0, 0), 40)
    h, w = f.shape
    energy[: crop // 2, :] = energy[h - crop // 2:, :] = 0          # keep crop inside image
    energy[:, : crop // 2] = energy[:, w - crop // 2:] = 0
    cy, cx = np.unravel_index(np.argmax(energy), energy.shape)
    c = f[cy - crop // 2: cy + crop // 2, cx - crop // 2: cx + crop // 2]
    c = c * np.outer(np.hanning(crop), np.hanning(crop))
    power = np.abs(np.fft.fftshift(np.fft.fft2(c))) ** 2
    yy, xx = np.indices(power.shape)
    r = np.hypot(yy - crop // 2, xx - crop // 2).astype(int)
    radial = np.bincount(r.ravel(), power.ravel()) / np.maximum(np.bincount(r.ravel()), 1)
    return radial * np.arange(len(radial))                          # compensate 1/f fall-off


# Plausible cell centre-to-centre distance (px). The first audit run found a clear
# cluster at 24–32 px; values outside this window are FFT misfires (artefacts, blur).
MIN_CELL_PX, MAX_CELL_PX = 18, 45


def cell_spacing_px(img, crop=CROP, min_px=MIN_CELL_PX, max_px=MAX_CELL_PX):
    """
    Cell centre-to-centre distance from the spectral ring. Returns (distance_px, reliable);
    `reliable` is False when the peak sits on the edge of the search window, i.e. the
    true lattice frequency was not found inside the plausible range.
    """
    radial = radial_power(img, crop)
    # hexagonal lattice: row spacing = d * sqrt(3) / 2  ->  ring radius k = crop / row_spacing
    k_lo = int(np.floor(crop / (max_px * np.sqrt(3) / 2)))
    k_hi = int(np.ceil(crop / (min_px * np.sqrt(3) / 2)))
    k = k_lo + int(np.argmax(radial[k_lo:k_hi + 1]))
    reliable = k_lo < k < k_hi
    return 2 * (crop / k) / np.sqrt(3), reliable


sample = inc.sample(min(FFT_SAMPLE, len(inc)), random_state=SEED)
sample = pd.concat([sample, inc[inc["split"] == "calibration"]]).drop_duplicates("image_id")
fft_results = [cell_spacing_px(load_gray(f)) for f in tqdm(sample["image_file"], desc="FFT")]
sample["cell_spacing_px"] = [d for d, _ in fft_results]
sample["fft_reliable"] = [ok for _, ok in fft_results]
n_unreliable = int((~sample["fft_reliable"]).sum())
print(f"FFT peak outside {MIN_CELL_PX}–{MAX_CELL_PX} px (unreliable, excluded): "
      f"{n_unreliable} of {len(sample)}")
sample = sample[sample["fft_reliable"]]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
example = load_gray(sample.iloc[0]["image_file"])
rp = radial_power(example)
axes[0].plot(np.arange(len(rp))[5:CROP // 4], rp[5:CROP // 4])
axes[0].set(xlabel="spatial frequency (cycles / crop)", ylabel="power × f",
            title=f"Radial spectrum, {sample.iloc[0]['image_id']}")
sns.histplot(sample["cell_spacing_px"], bins=30, ax=axes[1])
axes[1].set(xlabel="cell centre-to-centre distance (px)", title="Estimated cell size")
save(fig, "09_cell_spacing")

spacing_med = float(sample["cell_spacing_px"].median())
print(f"Median cell centre-to-centre distance: {spacing_med:.1f} px "
      f"(IQR {sample['cell_spacing_px'].quantile(.25):.1f}–{sample['cell_spacing_px'].quantile(.75):.1f})")
print(f"A 256×256 crop holds roughly {(256 / spacing_med) ** 2 * 2 / np.sqrt(3):.0f} cells")

cal = sample[sample["split"] == "calibration"].copy()
cal["d_um"] = np.sqrt(2 * cal["mean_area"] / np.sqrt(3))
cal["um_per_px_est"] = cal["d_um"] / cal["cell_spacing_px"]
um_per_px_prelim = float(cal["um_per_px_est"].median())
print(f"PRELIMINARY µm/px (calibration split, n={len(cal)}): {um_per_px_prelim:.3f} "
      f"(IQR {cal['um_per_px_est'].quantile(.25):.3f}–{cal['um_per_px_est'].quantile(.75):.3f}) "
      "— final value comes from Phase 4")

## 10 · Audit summary

In [ ]:
audit = {
    "counts": {k: int(v) for k, v in counts.items()},
    "ground_truth": {c: {"mean": round(float(inc[c].mean()), 2), "sd": round(float(inc[c].std()), 2),
                         "min": float(inc[c].min()), "max": float(inc[c].max())} for c in BIOMARKERS},
    "share_ecd_below_2000": round(float((inc["ecd"] < 2000).mean()), 4),
    "share_hex_below_60": round(float((inc["hex"] < 60).mean()), 4),
    "ecd_vs_inverse_mean_area_median_rel_diff": round(float(ecd_area_diff.median()), 4),
    "cv_hex_pearson_r": round(float(r_cv_hex), 3),
    "ecd_age_slope_per_year": round(float(age_fit.slope), 2),
    "grade_counts": {str(k): int(v) for k, v in inc["grade"].value_counts().items()},
    "ks_train_vs_test_pvalues": ks_pvalues,
    # Exploratory only — not used downstream (see Section 7 conclusion).
    "stress_line_exploratory": {
        "status": "unreliable; artifacts to be learned from expert patch labels (3.1)",
        "sigmas_half_res": list(RIDGE_SIGMAS), "min_line_length_px": MIN_LINE_LENGTH_PX,
        "min_total_px": STRESS_LINE_MIN_TOTAL_PX,
        "ridge_low": round(float(RIDGE_LOW), 5), "ridge_high": round(float(RIDGE_HIGH), 5),
        "flagged_share": round(float(inc["stress_line_flag"].mean()), 4),
        "visual_agreement": visual_agreement,
    },
    "fft_window_px": [MIN_CELL_PX, MAX_CELL_PX],
    "fft_unreliable_count": n_unreliable,
    "median_cell_spacing_px": round(spacing_med, 2),
    "preliminary_um_per_px_calibration_split": round(um_per_px_prelim, 4),
}
(OUT / "audit_summary.json").write_text(json.dumps(audit, indent=2))
print(json.dumps(audit, indent=2))
print("\nSaved to", OUT)

## Findings (draft for the thesis *Dataset* section)
*Numbers from the full runs of 2026-10-08 (seed 42, frozen split hash 5c739a9c…).*

**Dataset.** 1,389 specular images (1296 × 972, grayscale, Konan CellChek D, NEBSL). 1,384 are usable
(1,379 tissues; 5 tissues with two captures); 3 images have no Konan report and 2 uncertain matches are
held out. Frozen tissue-level split: train 921 · calibration 69 · val 197 · test 197. Train and test do
not differ in ECD, CV or HEX (KS p = 0.34 / 0.24 / 0.75).

**Ground truth.** ECD 2,757 ± 381 cells/mm² (range 1,010–4,255; 2.5% below 2,000), CV 36.4 ± 4.1%,
HEX 56.7 ± 6.1% (67% below the 60% healthy threshold), mean cell area 371 ± 65 µm². Konan analysed a
median of **110 cells per image** (47–277). ECD equals 10⁶ / mean area for every report (median difference
0.01%), so Konan ECD is derived from cell area — ECD and mean-area errors are effectively one quantity.
ECD falls by 8.3 cells/mm² per year of donor age (≈ 0.3%/yr, r = −0.26) and CV correlates with HEX
(r = −0.55), both consistent with the literature — supporting correct image–report matching.

**Image quality.** Specular images show strong, variable illumination fall-off (bright centre, dark
periphery). Whole-image quality scores penalise this even when the central mosaic is sharp (the
"worst" images by score contain crisp cells), so image quality must be judged **locally**, motivating
patch-based automatic ROI selection (Sub-component 3.1).

**Artifacts.** Stress lines and folds are visibly common, but a classical ridge detector could not
separate them reliably from illumination boundaries (≈ 55% visual agreement, 39% of images flagged).
Artifact detection is therefore learned from expert patch labels (3.1); its prevalence will be
measured from the Phase 1 annotation.

**Cell size.** FFT lattice analysis gives a median cell centre-to-centre distance of **28.2 px**
(range 23–42 px; 10 of ≈ 250 sampled images had no clear lattice peak and were excluded). A 256 × 256
crop therefore holds ≈ 95 cells — suitable U-Net training tiles at full resolution; downscaling by 2
would leave ≈ 1-px cell walls. Preliminary scale (calibration split only, n = 69): **≈ 0.75 µm/px**
(final value: Phase 4).

**Coverage (preliminary).** At ≈ 0.75 µm/px one image covers ≈ 0.71 mm², i.e. ≈ 1,900 cells at the
mean ECD; Konan's ≈ 110 cells (≈ 0.04 mm²) cover only ≈ 5–6% of the image (Gap 3).

**Limitations.** 1,382 of 1,384 images are Konan grade *green* (little low-quality data); acquisition
years are skewed to 2020 (563 images); eye-to-donor grouping is unconfirmed (tissues grouped by serial).